# Seasonal ARIMA (SARIMA): lab notebook

**Statistical Modelling and Forecasting, ARIMA and vector autoregression**

Run the cells from top to bottom (Runtime > Run all). NumPy, pandas, SciPy, statsmodels and scikit-learn are already installed in Colab. Read each section, run it, and compare your output with the expected output at the end of the notebook.

In [ ]:
import warnings
warnings.filterwarnings("ignore")  # keeps the output tidy

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.tsa.stattools import acf

### 1. Monthly atmospheric CO2 (weekly data averaged by month)

In [ ]:
# 1. Monthly atmospheric CO2 (weekly data averaged by month)
y = sm.datasets.co2.load_pandas().data["co2"].resample("MS").mean().interpolate()
y = y["1975":]
print("observations:", len(y), "| from", y.index[0].date(), "to", y.index[-1].date())

### 2. Differencing: one ordinary and one seasonal difference

In [ ]:
# 2. Differencing: one ordinary and one seasonal difference
w = y.diff().diff(12).dropna()
print("observations after (1-B)(1-B^12):", len(w), "= n - 13 =", len(y) - 13)
r = acf(w, nlags=26)
print("ACF of the differenced series at lags 1, 2, 11, 12, 13, 24:", np.round(r[[1, 2, 11, 12, 13, 24]], 2))

### 3. Fit the airline model ARIMA(0,1,1)(0,1,1)12 and a competitor

In [ ]:
# 3. Fit the airline model ARIMA(0,1,1)(0,1,1)12 and a competitor
train, test = y.iloc[:-24], y.iloc[-24:]
m1 = SARIMAX(train, order=(0, 1, 1), seasonal_order=(0, 1, 1, 12)).fit(disp=False)
m2 = SARIMAX(train, order=(1, 1, 1), seasonal_order=(0, 1, 1, 12)).fit(disp=False)
print(m1.summary().tables[1])
print("AIC (0,1,1)(0,1,1)12:", round(m1.aic, 1), "| AIC (1,1,1)(0,1,1)12:", round(m2.aic, 1))
print("MA product term at lag 13 = theta*Theta =", round(m1.params["ma.L1"] * m1.params["ma.S.L12"], 3))

### 4. Residual check

In [ ]:
# 4. Residual check
lb = acorr_ljungbox(m1.resid.iloc[13:], lags=[24], model_df=2)
print("Ljung-Box p-value (24 lags):", round(float(lb["lb_pvalue"].iloc[0]), 3))

### 5. Forecast two years and compare with the seasonal naive forecast

In [ ]:
# 5. Forecast two years and compare with the seasonal naive forecast
fc = m1.get_forecast(24)
mean = fc.predicted_mean
ci = fc.conf_int()
snaive = train.iloc[-12:].values.tolist() * 2
rmse = lambda e: float(np.sqrt(np.mean(np.square(e))))
print("RMSE SARIMA:", round(rmse(test.values - mean.values), 3), "| RMSE seasonal naive:", round(rmse(test.values - np.array(snaive)), 3))
print("first 3 forecasts:", np.round(mean.iloc[:3].values, 2), "| 95% interval width at h = 1 and 24:", np.round([ci.iloc[0, 1] - ci.iloc[0, 0], ci.iloc[-1, 1] - ci.iloc[-1, 0]], 2))

### 6. The numerical problem (s = 4): w_T and the forecast

In [ ]:
# 6. The numerical problem (s = 4): w_T and the forecast
yv = [60, 66, 72, 62, 64, 71]
print("w_T =", yv[5] - yv[4] - yv[1] + yv[0])
th, TH, eT, e3, e4 = -0.4, -0.5, 1, -1, 2
print("forecast:", yv[5] + yv[2] - yv[1] + th * eT + TH * e3 + th * TH * e4)

## Expected output

Running all cells above should print:

```
observations: 324 | from 1975-01-01 to 2001-12-01
observations after (1-B)(1-B^12): 311 = n - 13 = 311
ACF of the differenced series at lags 1, 2, 11, 12, 13, 24: [-0.32 -0.06  0.16 -0.4   0.04 -0.04]
==============================================================================
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
ma.L1         -0.3788      0.051     -7.485      0.000      -0.478      -0.280
ma.S.L12      -0.8630      0.045    -19.302      0.000      -0.951      -0.775
sigma2         0.0790      0.006     12.290      0.000       0.066       0.092
==============================================================================
AIC (0,1,1)(0,1,1)12: 108.5 | AIC (1,1,1)(0,1,1)12: 110.5
MA product term at lag 13 = theta*Theta = 0.327
Ljung-Box p-value (24 lags): 0.794
RMSE SARIMA: 0.438 | RMSE seasonal naive: 2.066
first 3 forecasts: [369.02 369.83 370.77] | 95% interval width at h = 1 and 24: [1.1  3.71]
w_T = 1
forecast: 77.5
```

## Practice exercises

Try these in the cells below.

1. Fit (0,1,1)(0,1,1) to the monthly series, then drop the seasonal MA. Compare residual autocorrelation at lag 12.
2. Fit the same orders with D = 0. What happens to the forecast pattern?
3. Increase the horizon to 36 months and describe how the interval grows.

In [ ]:
# Exercise 1


In [ ]:
# Exercise 2


In [ ]:
# Exercise 3
